In [6]:
import os
import re
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from google import genai

load_dotenv()                                    # reads GEMINI_API_KEY from .env
assert os.getenv("GEMINI_API_KEY"), "Add GEMINI_API_KEY to your .env file"

client = genai.Client()                          # picks the key up from the environment

# Model names change over time. If one fails, copy the current name from Google's Gemini API docs.
EMBED_MODEL = "gemini-embedding-001"
DB_PATH = "chroma_db"                            # folder where the database is saved
COLLECTION_NAME = "notes"

In [7]:
def chunk_paragraphs(text, max_chars=500):
    """Split text into paragraph chunks, each at most `max_chars` long."""
    chunks = []
    for para in re.split(r"\n\s*\n", text):
        para = para.strip()
        if not para:
            continue
        if len(para) <= max_chars:
            chunks.append(para)
            continue
        current = ""
        for sentence in re.split(r"(?<=[.!?])\s+", para):
            if current and len(current) + 1 + len(sentence) > max_chars:
                chunks.append(current)
                current = sentence
            else:
                current = f"{current} {sentence}".strip()
        if current:
            chunks.append(current)
    return chunks

In [8]:
SAMPLE_NOTES = {
    "dbms.txt": """
Normalization is the process of organizing a database to reduce duplicate data. We split big tables into smaller ones and link them with keys, so a change only has to be made in one place.

A join combines rows from two tables using a related column. An inner join keeps only the rows that match in both tables. A left join keeps all rows from the left table, even when there is no match on the right.

ACID describes the four properties of a reliable transaction: Atomicity, Consistency, Isolation and Durability. Atomicity means a transaction either completes fully or not at all.
""",
    "os.txt": """
A deadlock happens when two or more processes wait for each other forever. Process A holds resource 1 and needs resource 2, while process B holds resource 2 and needs resource 1. Neither can continue, so both are stuck.

A process is a running program with its own memory space. A thread is a smaller unit of execution inside a process. Threads of the same process share memory, so they are lighter than processes but need careful synchronization.

Paging splits memory into fixed-size blocks called pages. Virtual memory lets a program use more memory than is physically installed by keeping some pages on disk.
""",
    "python.txt": """
A list is an ordered collection that you can change after creating it. A tuple is also ordered, but it is immutable, so you cannot change it after it is created.

A dictionary stores key-value pairs. You look up a value by its key, for example scores["asha"], and lookups are fast because keys are hashed.

Exceptions are handled with try and except blocks. Code that might fail goes in try, and the recovery code goes in except. A finally block runs whether or not an error happened.
""",
}

# Use your own notes if you have a notes/ folder with .txt files, otherwise the samples
NOTES_DIR = Path("notes")
if NOTES_DIR.exists() and any(NOTES_DIR.glob("*.txt")):
    notes = {p.name: p.read_text(encoding="utf-8") for p in sorted(NOTES_DIR.glob("*.txt"))}
else:
    notes = SAMPLE_NOTES

records = []
for filename, text in notes.items():
    for n, chunk in enumerate(chunk_paragraphs(text), start=1):
        records.append({"id": f"{filename}-{n}", "source": filename, "chunk_no": n, "text": chunk})

print(len(records), "chunks from", len(notes), "files")

9 chunks from 3 files


In [9]:
def embed(texts, batch_size=50):
    """Turn a list of texts into a list of vectors (lists of numbers)."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        result = client.models.embed_content(model=EMBED_MODEL, contents=batch)
        vectors.extend(e.values for e in result.embeddings)
    return vectors


test_vec = embed(["What is a deadlock?"])[0]
print("vector length:", len(test_vec))
print("first 5 numbers:", test_vec[:5])

vector length: 3072
first 5 numbers: [0.013243918, 0.002794176, -0.0136028305, -0.054120217, 0.02035944]


In [10]:
chroma = chromadb.PersistentClient(path=DB_PATH)   # saves to the chroma_db/ folder

try:
    chroma.delete_collection(COLLECTION_NAME)      # start fresh so re-running doesn't duplicate
except Exception:
    pass
collection = chroma.create_collection(COLLECTION_NAME)

collection.add(
    ids=[r["id"] for r in records],
    documents=[r["text"] for r in records],
    embeddings=embed([r["text"] for r in records]),
    metadatas=[{"source": r["source"], "chunk_no": r["chunk_no"]} for r in records],
)

print("stored", collection.count(), "chunks in", DB_PATH)

stored 9 chunks in chroma_db


In [11]:
peek = collection.peek(3)
for doc_id, doc in zip(peek["ids"], peek["documents"]):
    print(doc_id, "->", doc[:70], "...")

dbms.txt-1 -> Normalization is the process of organizing a database to reduce duplic ...
dbms.txt-2 -> A join combines rows from two tables using a related column. An inner  ...
dbms.txt-3 -> ACID describes the four properties of a reliable transaction: Atomicit ...
